In [9]:
import pandas as pd
from pyomo.environ import *

CN_FLB_SCP

In [ ]:
# Read the excel file into a pandas DataFrame, change the file path to your own 
# file_name_FL2SCP_CN = "C:\\Workspace\\Food_lost2Ch\\IO\\FL2SCP_CN_ADP.xlsx" #scenario APD
## or
file_name_FL2SCP_CN = "C:\\Workspace\\Food_lost2Ch\\IO\\FL2SCP_CN_TP.xlsx" #scenario TP


# eta is yield
eta_df = pd.read_excel(file_name_FL2SCP_CN, sheet_name = 'eta', index_col=0)

# FL is the qty of each Food loss or by-products stream
FL_df=pd.read_excel(file_name_FL2SCP_CN, sheet_name = 'FL')

# SCP is the conversion rate of SCP to edible protein
SCP_df=pd.read_excel(file_name_FL2SCP_CN, sheet_name = 'SCP')

# GWP is the global warming potential saving of each FL_SCP pathway
GWP_df=pd.read_excel(file_name_FL2SCP_CN, sheet_name = 'GWP')

# WU is the water use saving of each FL_SCP pathway
WU_df=pd.read_excel(file_name_FL2SCP_CN, sheet_name = 'WU')

# LU is the land use saving of each FL_SCP pathway
LU_df=pd.read_excel(file_name_FL2SCP_CN, sheet_name = 'LU')

In [11]:
# Convert the DataFrame to a dictionary
eta_dict = eta_df.to_dict(orient='index')
FL_dict = FL_df['Column1'].to_dict()
SCP_dict = SCP_df['Column1'].to_dict()
WU_dict = WU_df['Column1'].to_dict()
GWP_dict = GWP_df['Column1'].to_dict()
LU_dict = LU_df['Column1'].to_dict()

# Convert nested dictionary format for Pyomo
eta_dict = {(row, col): value for row, row_data in eta_dict.items() for col, value in row_data.items()}
FL_dict = {i+1: FL_dict[i] for i in FL_dict}
SCP_dict = {i+1: SCP_dict[i] for i in SCP_dict}
WU_dict = {i+1: WU_dict[i] for i in WU_dict}
GWP_dict = {i+1: GWP_dict[i] for i in GWP_dict}
LU_dict = {i+1: LU_dict[i] for i in LU_dict}

print(GWP_dict)

{1: 45838653.00000001, 2: -51593737.76153295, 3: -55909970.28153295, 4: 10257554.494179646, 5: 5941321.974179645, 6: 46838653.00000001, 7: 10768312.52, 8: 6452079.999999999, 9: 49297572.31902295, 10: 9006355.79229851, 11: 4690123.272298509}


In [12]:
# Create a Concrete Model
model = ConcreteModel()

# Sets
model.I = RangeSet(1, 22) # for CN, (1, 22), for US (1, 13), for EU (1, 19)
model.J = RangeSet(1, 11) # 11 FLB_SCP_Food/feed pathways


# Parameters
model.FL = Param(model.I, initialize=FL_dict) 
model.SCP = Param(model.J, initialize=SCP_dict)  #conversion rate (SCP to human edible protein)
model.eta = Param(model.I, model.J, initialize=eta_dict)

#Define the parameters of objective function, make one choice 
# #option 1: model WU,choose this option when run the objective function 1
# model.WU = Param(model.J, initialize=WU_dict)  
# # option 2: model GWP, choose this option when run the objective function 2
# model.GWP = Param(model.J, initialize=GWP_dict)
#option 3: model LU,choose this option when run the objective function 3
model.LU = Param(model.J, initialize=LU_dict)  

# upper limit of total SCP production, SCP_max_value, unit kiloton, for Animsl derieved protein scenario: 15203.12242; for total protein scenario:66563.58329.
model.SCP_max = Param(initialize=66563.58329)

# Variables
model.x = Var(model.I, within=NonNegativeReals) # the quantity of FLB used
model.y = Var(model.J, within=NonNegativeReals) # the SCP produced per pathway
model.w = Var(model.I, model.J, within=NonNegativeReals) # w[i,j] indicate the FL[i] used in pathway FLB_SCP[j]

# # option 1: Objective function 1, WU saving
# def objective_rule(model):
#     return sum(model.y[j] * model.WU[j] for j in model.J)

# # option 2: Objective function 2, GWP saving
# def objective_rule(model):
#     return sum(model.y[j] * model.GWP[j] for j in model.J)

# option 3: Objective function 3, LU saving
def objective_rule(model):
    return sum(model.y[j] * model.LU[j] for j in model.J)

model.obj = Objective(rule=objective_rule, sense=maximize)

# Constraints
def x_constraint_rule(model, i):
    return model.x[i] == sum(model.w[i, j] for j in model.J)
model.x_constraint = Constraint(model.I, rule=x_constraint_rule)

def y_constraint_rule(model, j):
    return model.y[j] == sum(model.w[i, j] * model.eta[i, j] for i in model.I)*model.SCP[j]
model.y_constraint = Constraint(model.J, rule=y_constraint_rule)

def total_scp_constraint_rule(model):
    return sum(model.y[j] for j in model.J) <= model.SCP_max
model.total_scp_constraint = Constraint(rule=total_scp_constraint_rule)

def x_fleet_constraint_rule(model, i):
    return model.x[i] <= model.FL[i]
model.x_fleet_constraint = Constraint(model.I, rule=x_fleet_constraint_rule)

# Solve the model
solvername='glpk'
solverpath_folder='C:\Python311\winglpk\glpk-4.65\w64' #does not need to be directly on c drive
solverpath_exe='C:\Python311\winglpk\glpk-4.65\w64\glpsol.exe' #does not need to be directly on c drive
# solver = SolverFactory('glpk')

solver=SolverFactory(solvername,executable=solverpath_exe)

results = solver.solve(model)

# Display the results
model.display()




Model unknown

  Variables:
    x : Size=22, Index=I
        Key : Lower : Value            : Upper : Fixed : Stale : Domain
          1 :     0 : 10144.2341500138 :  None : False : False : NonNegativeReals
          2 :     0 : 15996.6655791288 :  None : False : False : NonNegativeReals
          3 :     0 : 38142.4419027004 :  None : False : False : NonNegativeReals
          4 :     0 : 19071.2209513502 :  None : False : False : NonNegativeReals
          5 :     0 : 8336.85548147133 :  None : False : False : NonNegativeReals
          6 :     0 : 17385.9370767711 :  None : False : False : NonNegativeReals
          7 :     0 : 2634.23289041986 :  None : False : False : NonNegativeReals
          8 :     0 : 15189.6161458487 :  None : False : False : NonNegativeReals
          9 :     0 :          27000.0 :  None : False : False : NonNegativeReals
         10 :     0 : 2333.33333333333 :  None : False : False : NonNegativeReals
         11 :     0 : 17582.6813242053 :  None : False 